In [246]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns


In [247]:
dx_df = pd.read_csv('data/DXSUM_17Feb2026.csv')
dem  = pd.read_csv('data/PTDEMOG_17Feb2026.csv') 
adas = pd.read_csv('data/ADAS_17Feb2026.csv')
medhist = pd.read_csv('data/MEDHIST_17Feb2026.csv')
recmhist = pd.read_csv('data/RECMHIST_17Feb2026.csv')
demographics = pd.read_csv('data/PTDEMOG_17Feb2026.csv')
apoe_df = pd.read_csv("data/APOERES_17Feb2026.csv")
apoe_df["CARRIER"] = apoe_df["GENOTYPE"].isin(["2/4","3/4","4/4"])
apoe_df["HOMO"] = apoe_df["GENOTYPE"].isin(["4/4"])

In [248]:
def visit_to_months(v):
    if pd.isna(v):
        return np.nan
    v = v.lower()
    if v in ["bl", "sc"]:
        return 0
    if v.startswith("m"):
        return float(v[1:])  # strip the 'm'
    if v.startswith("v"):
        return float(v[1:])  # strip the 'v'
    
    return np.nan

baseline_codes = ["bl", "4_bl"]
adas["time_months"] = adas["VISCODE"].apply(visit_to_months)

baseline_dates = (
    adas[adas["VISCODE"].isin(baseline_codes)]
    .groupby("RID")["VISDATE"]
    .min()
)

adas_df = adas.merge(
    baseline_dates.rename("baseline_date"),
    on="RID",
    how="left"
)

adas_df["days_since_bl"] = (
    pd.to_datetime(adas_df["VISDATE"]) -
    pd.to_datetime(adas_df["baseline_date"])
)

In [249]:
# Merge ADAS scores with diagnosis data
adas_AD_dx = pd.merge(adas_df, dx_df[["RID", "VISCODE", "DIAGNOSIS"]], on=["RID", "VISCODE"], how="left")
adas_AD_dx['VISDATE'] = pd.to_datetime(adas_AD_dx['VISDATE'])


### Finding comorbidities with string matching:

In [250]:
UTI_KEYWORDS = [
    "urinary tract infection",
    "uti",
    "bladder infection",
    "cystitis",
    "urosepsis",
    "pyelonephritis",
    "urinary infection",
    "recurrent uti",
    # GI KEYWORDS:
    "halitosis",
    "diarrhea",
    "constipation",
    #"bowel",
    #"gastrointestinal",
    #"gi symptoms",
    #"fecal incontinence",
    #"fecal impaction",
    #"fecal retention",
    #"fecal urgency",
    #"faecal incontinence",

]

Z867_KEYWORDS = [
    
    "cardiac arrest",
    "stroke",
    "ischemic",
    "phebitis",
    "thrombophlebitis",
    "thromboembolism",
    "pulmunary embolism",

]

Z864_KEYWORDS = [
    "alcohol abuse",
    "alcohol dependence",
    "alcoholism",
    "substance abuse",
    "drug abuse",
    "drug dependence",
    "opioid dependence",
    "cocaine use",
    "heroin use",
    "illicit drug use",
    "history of substance abuse"
]

SLEEP_KEYWORDS = [
    "insomnia",
    "sleep disorder",
    "sleep disturbance",
    "poor sleep",
    "sleep apnea",
    "obstructive sleep apnea",
    "osa",
    "hypersomnia",
    "sleep fragmentation",
    "restless legs",
    "bruxism",
    "parasomnia",
    "narcolepsy",
    
]

DENTAL_KEYWORDS = [
    "periodontitis",
    "gingivitis",
    "gum disease",
    "dental abscess",
    "tooth infection",
    "tooth loss",
    "edentulous",
    "poor dentition",
    "dental caries",
    "tooth decay"
]

ANXIETY_KEYWORDS = [
    "anxiety",
    "anxious",
    "anxiety disorder",
    "generalised anxiety",
    "generalized anxiety",
    "gad",
    "panic disorder",
    "panic attack",
    "agoraphobia",
    "social anxiety",
    "post-traumatic stress",
    "posttraumatic stress",
    "ptsd",
    "obsessive compulsive",
    "ocd",
    "nervousness",
    "worry disorder",
#]
#
#DEPRESSION_KEYWORDS = [
    "depression",
    "depressive",
    "depressed",
    "major depressive disorder",
    "mdd",
    "dysthymia",
    "dysthymic",
    "bipolar",  
    "low mood",
    "anhedonia",
    "mood disorder",
    "affective disorder",
    "seasonal affective",
    "antidepressant",  
    "persistent depressive"
]

In [251]:
T2_DIABETES_KEYWORDS = [
    "type 2 diabetes",
    "type ii diabetes",
    "t2dm",
    "t2d",
    "diabetes mellitus type 2",
    "non-insulin dependent diabetes",
    "niddm",
    "adult onset diabetes",
]

ATRIAL_FIBRILLATION_KEYWORDS = [
    "atrial fibrillation",
    "atrial flutter",
    "afib",
    "a-fib",
    "a fib",
    "paroxysmal atrial fibrillation",
    "persistent atrial fibrillation",
    "permanent atrial fibrillation",
    "chronic atrial fibrillation",
    "auricular fibrillation",
]

HEART_FAILURE_KEYWORDS = [
    "heart failure",
    "cardiac failure",
    "congestive heart failure",
    "systolic heart failure",
    "diastolic heart failure",
    "left ventricular failure",
    "right ventricular failure",
    "left heart failure",
    "right heart failure",
    "hfref",
    "hfpef",
    "cardiomyopathy",
    "dilated cardiomyopathy",
    "ischemic cardiomyopathy",
]

ISCHEMIC_HEART_DISEASE_KEYWORDS = [
    "ischemic heart disease",
    "ischaemic heart disease",
    "ihd",
    # Add in Z867 and chest pain keywords:
    "ischemic",
    "phebitis",
    "thrombophlebitis",
    "thromboembolism",
    "pulmunary embolism",
    "chest pain",
]

STROKE_KEYWORDS = [
    "stroke",
    "cerebrovascular accident",
    "cerebral infarction",
    "cerebral infarct",
    "ischemic stroke",
    "ischaemic stroke",
    "haemorrhagic stroke",
    "hemorrhagic stroke",
    "transient ischemic attack",
    "transient ischaemic attack",
    "brain attack",
]

In [252]:
SPECIFIC_KEYWORDS = {
    "UTI": UTI_KEYWORDS,
    "Z867": Z867_KEYWORDS,
    "Z864": Z864_KEYWORDS,
    "Sleep": SLEEP_KEYWORDS,
    "Dental": DENTAL_KEYWORDS,
    "Anxiety": ANXIETY_KEYWORDS,
    "CM_diabetes": T2_DIABETES_KEYWORDS,
    "CM_afib": ATRIAL_FIBRILLATION_KEYWORDS,
    "CM_heart_failure": HEART_FAILURE_KEYWORDS,
    "CM_ihd": ISCHEMIC_HEART_DISEASE_KEYWORDS,
    "CM_stroke": STROKE_KEYWORDS,}


def classify_specific(desc):
    if pd.isna(desc):
        return None
    
    d = desc.lower()
    d = d.replace("hx of", "history of")
    d = d.replace("h/o", "history of")
    
    for category, keywords in SPECIFIC_KEYWORDS.items():
        if any(k in d for k in keywords):
            return category
    
    return None

recmhist["specific_flag"] = recmhist["MHDESC"].apply(classify_specific)

In [253]:
CNS_KEYWORDS = {
    'cerebrovascular': ['stroke', 'cerebrovascular', 'tia', 'transient ischemic', 'cva'],
    'insomnia': ['insomnia', 'sleep disorder'],
    'anxiety': ['anxiety', 'anxious'],
    'depression': ['depression', 'depressive', 'depressed'],
    'head_injury': ['head injury', 'tbi', 'traumatic brain', 'concussion', 'head trauma']
}

PERIPHERAL_KEYWORDS = {
    'hypertension': ['hypertension', 'high blood pressure', 'htn'],
    'hyperlipidemia': ['hyperlipidemia', 'hypercholesterolemia', 'high cholesterol', 'dyslipidemia'],
    'diabetes': ['diabetes', 'diabetic', 'dm', 'niddm', 'iddm'],
    'atrial_fibrillation': ['atrial fibrillation', 'afib', 'a fib', 'a-fib'],
    'coronary': ['coronary', 'ischemic heart', 'ihd', 'cad', 'coronary artery', 'myocardial infarction', 'mi', 'heart attack'],
    'anemia': ['anemia', 'anaemia'],
    'hypothyroid': ['hypothyroid', 'thyroid'],
    'skin_inflammatory': ['psoriasis', 'eczema', 'dermatitis', 'skin inflammation'],
    'pulmonary': ['copd', 'asthma', 'pulmonary', 'emphysema', 'chronic bronchitis', 'respiratory', 'lung disease'],
    'kidney': ['kidney', 'renal', 'ckd', 'chronic kidney'],
    'hepatitis': ['hepatitis', 'liver disease', 'cirrhosis'],
    'osteoporosis': ['osteoporosis', 'bone density'],
    'hearing_loss': ['hearing loss', 'deaf', 'hearing impair'],
    'cancer': ['cancer', 'malignancy', 'carcinoma', 'tumor', 'neoplasm'],
    'gastrointestinal': ['gastro', 'ulcer', 'reflux', 'gerd', 'ibs', 'crohn', 'colitis', 'diverticulitis'],
    'cataract': ['cataract'],
}

def classify_condition_detailed(desc):
    """
    Classify condition into CNS, Peripheral, or Other
    Returns tuple: (category, specific_condition)
    """
    if pd.isna(desc):
        return None, None
    
    d = desc.lower()
    
    # Check CNS conditions
    for condition, keywords in CNS_KEYWORDS.items():
        if any(k in d for k in keywords):
            return "CNS", condition
    
    # Check Peripheral conditions
    for condition, keywords in PERIPHERAL_KEYWORDS.items():
        if any(k in d for k in keywords):
            return "Peripheral", condition
    
    return "Other", None

recmhist["comorb_category"] = recmhist["MHDESC"].apply(
    lambda x: classify_condition_detailed(x)[0]
)
recmhist["specific_condition"] = recmhist["MHDESC"].apply(
    lambda x: classify_condition_detailed(x)[1]
)

In [269]:
recmhist_bl = recmhist[recmhist["VISCODE"].isin(["v01", "sc"])]

In [270]:
comorb_counts = (
    recmhist_bl[recmhist_bl["comorb_category"].isin(["CNS", "Peripheral"])]
    .groupby(["RID", "VISCODE", "comorb_category", "specific_condition"])
    .size()
    .reset_index(name="count")
    .groupby(["RID", "VISCODE", "comorb_category"])
    .agg(
        n_conditions=("specific_condition", "nunique"),
        total_entries=("count", "sum")
    )
    .reset_index()
)

# Create wide format
comorb_wide = comorb_counts.pivot_table(
    index=["RID", "VISCODE"],
    columns="comorb_category",
    values="n_conditions",
    fill_value=0
).reset_index()

# Calculate total multimorbidity burden
comorb_wide["total_conditions"] = (
    comorb_wide.get("CNS", 0) + 
    comorb_wide.get("Peripheral", 0)
)

# Categorize burden as in the paper
def categorize_burden(n):
    if n <= 2:
        return "Low"
    elif n <= 5:
        return "Medium"
    else:
        return "High"

comorb_wide["burden_category"] = comorb_wide["total_conditions"].apply(categorize_burden)

# For CNS burden: 0 vs 1+ 
comorb_wide["CNS_burden"] = comorb_wide["CNS"].apply(lambda x: "0" if x == 0 else "1+")

# For Peripheral burden: 0-1 vs 2+ 
comorb_wide["Peripheral_burden"] = comorb_wide["Peripheral"].apply(
    lambda x: "0-1" if x <= 1 else "2+"
)

In [271]:
specific_flags_visit = (
    recmhist[recmhist["specific_flag"].notna()]
    .assign(flag=1)
    .pivot_table(
        index=["RID", "VISCODE"],
        columns="specific_flag",
        values="flag",
        aggfunc="max",
        fill_value=0
    )
    .reset_index()
)

comorb_wide_new = comorb_wide.merge(
    specific_flags_visit,
    on=["RID", "VISCODE"],
    how="left"
)

CM_COLS = ["CM_diabetes", "CM_afib", "CM_heart_failure", "CM_ihd", "CM_stroke"]

# Replace NaNs with 0s for the specific flags
for col in ["UTI", "Z867", "Z864", "Sleep", "Dental", "Anxiety"] + CM_COLS:
    if col in comorb_wide_new.columns:
        comorb_wide_new[col] = comorb_wide_new[col].fillna(0)

comorb_wide_new["CM"] = (
    comorb_wide_new[[c for c in CM_COLS if c in comorb_wide_new.columns]]
    .max(axis=1)
    .clip(upper=1)
    .astype(int)
)

In [274]:
demographics_unique = demographics[["RID", "PTDOB", "PTGENDER", "PTEDUCAT"]].drop_duplicates(subset=["RID"])

AD_dem = adas_AD_dx.merge(

    demographics_unique,

    on="RID",

    how="left",

    validate="many_to_one"

)

print(f"Number of individuals with demographic data and cognitive data: {AD_dem['RID'].nunique()}")


print(comorb_wide_new.columns)
print(comorb_wide_new.head())
comorb_visit = comorb_wide_new[[

    "RID", "total_conditions", "CNS", "Peripheral",

    "UTI", "Z867", "Z864", "Sleep", "Dental", "Anxiety", "CM"

]].drop_duplicates(subset=["RID"])



AD = AD_dem.merge(

    comorb_visit,

    on="RID",

    how="left",

    validate="many_to_one"

)



print(f"Number of individuals with comorbidity data: {AD['RID'].nunique()}" )

Number of individuals with demographic data and cognitive data: 3027
Index(['RID', 'VISCODE', 'CNS', 'Peripheral', 'total_conditions',
       'burden_category', 'CNS_burden', 'Peripheral_burden', 'Anxiety',
       'CM_afib', 'CM_diabetes', 'CM_heart_failure', 'CM_ihd', 'CM_stroke',
       'Dental', 'Sleep', 'UTI', 'Z864', 'Z867', 'CM'],
      dtype='object')
   RID VISCODE  CNS  Peripheral  total_conditions burden_category CNS_burden  \
0    2      sc  0.0         2.0               2.0             Low          0   
1    3      sc  1.0         3.0               4.0          Medium         1+   
2    4      sc  1.0         1.0               2.0             Low         1+   
3    5      sc  0.0         2.0               2.0             Low          0   
4    6      sc  0.0         4.0               4.0          Medium          0   

  Peripheral_burden  Anxiety  CM_afib  CM_diabetes  CM_heart_failure  CM_ihd  \
0                2+      0.0      0.0          0.0               0.0     0.0  

In [275]:
ad_start = AD[['RID', 'VISCODE', 'VISDATE', 'TOTAL13', 'PTGENDER', 'PTDOB', 'PTEDUCAT']].copy()
first_entries = (
    ad_start
    .sort_values('VISDATE')
    .groupby('RID')
    .first()
    .reset_index()
)
print(first_entries['VISCODE'].value_counts())
first_entries['VISDATE'] = pd.to_datetime(first_entries['VISDATE'])
first_entries['PTDOB'] = pd.to_datetime(first_entries['PTDOB'])
# Calculate age at conversion (in years)
first_entries['age_at_baseline'] = (
    (first_entries['VISDATE'] - first_entries['PTDOB'])
    .dt.days / 365.25
)

ad_start = first_entries.rename(columns={
    'VISDATE': 'Study_start_date',
    'TOTAL13': 'TOTAL13_AD_start'
})

AD_new = AD.merge(ad_start[['RID', 'Study_start_date', 'TOTAL13_AD_start', 'age_at_baseline']], on='RID', how='left')
print(AD_new["RID"].nunique())


VISCODE
bl        1647
v03        788
4_bl       589
4_init       3
Name: count, dtype: int64
3027


/var/folders/g_/qzn_bmsd7v9fwp049f83fwtr0000gp/T/ipykernel_31484/3794980079.py:11: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  first_entries['PTDOB'] = pd.to_datetime(first_entries['PTDOB'])


In [276]:
model_vars = [
    "TOTAL13",
    "TOTAL13_AD_start",
    "VISCODE",
    "VISDATE",
    "PHASE",
    "days_since_entry",
    "age_at_baseline",
    "PTGENDER",
    "CARRIER",
    "HOMO",
    "PTEDUCAT",
    "CNS",
    "total_conditions",
    "Peripheral",
    "RID",
    "UTI", "Sleep", "CM"
]

In [277]:
AD_df = AD_new.merge(apoe_df[["RID", "CARRIER", "HOMO"]], on="RID", how="left")
print(AD_df["RID"].nunique())
AD_df['VISDATE'] = pd.to_datetime(AD_df['VISDATE'])
AD_df['Study_start_date'] = pd.to_datetime(AD_df['Study_start_date'])
AD_df['days_since_entry'] = (AD_df['VISDATE'] - AD_df['Study_start_date']).dt.days

AD_no_EO = AD_df[AD_df['age_at_baseline'] >= 65]

AD_model = AD_no_EO[model_vars].dropna()

AD_model["age_c"] = AD_model["age_at_baseline"] - AD_no_EO["age_at_baseline"].mean()
AD_model["edu_c"] = AD_model["PTEDUCAT"] - AD_no_EO["PTEDUCAT"].mean()
AD_model["time_years"] = AD_model["days_since_entry"] / 365.25

print(AD_model["RID"].nunique())
print(AD_model.head())

3027
1469
    TOTAL13  TOTAL13_AD_start VISCODE    VISDATE  PHASE  days_since_entry  \
3     17.00             13.67     m06 2008-03-25  ADNI1             230.0   
4     13.67             13.67      bl 2007-08-08  ADNI1               0.0   
8      2.00              4.00     m24 2009-10-30  ADNI1             821.0   
9      0.00              4.00     m12 2008-07-30  ADNI1             364.0   
10     3.00              4.00     m06 2008-02-06  ADNI1             189.0   

    age_at_baseline  PTGENDER CARRIER   HOMO  ...  CNS  total_conditions  \
3         74.269678       1.0   False  False  ...  0.0               4.0   
4         74.269678       1.0   False  False  ...  0.0               4.0   
8         71.581109       1.0   False  False  ...  0.0               3.0   
9         71.581109       1.0   False  False  ...  0.0               3.0   
10        71.581109       1.0   False  False  ...  0.0               3.0   

    Peripheral   RID  UTI  Sleep   CM     age_c     edu_c  time_years 

### Add in CSF data:

In [278]:
CSF = pd.read_csv("data/UPENNBIOMK_ROCHE_ELECSYS_19Feb2026.csv")

CSF["PTAU_ABETA42"] = CSF["PTAU"] / CSF["ABETA42"]
CSF["PT_AB_std"] = (CSF["PTAU_ABETA42"] - CSF["PTAU_ABETA42"].mean()) / CSF["PTAU_ABETA42"].std()

CSF_bl = CSF[CSF["VISCODE2"] == "bl"]
CSF_bl["CSF_date"] = CSF_bl["EXAMDATE"]

/var/folders/g_/qzn_bmsd7v9fwp049f83fwtr0000gp/T/ipykernel_31484/375718845.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  CSF_bl["CSF_date"] = CSF_bl["EXAMDATE"]


In [279]:
CSF_AD = AD_model.merge(
    CSF_bl[["RID", "VISCODE2", "ABETA42", "TAU", "PTAU", "PTAU_ABETA42", "PT_AB_std", "CSF_date"]],
    on=["RID"],
    how="left")
CSF_AD["time_sq"] = CSF_AD["time_years"] ** 2
CSF_AD = CSF_AD.dropna(subset=["TOTAL13"])
CSF_AD = CSF_AD.dropna(subset=["PT_AB_std"])
CSF_AD["baseline_c"] = (
CSF_AD["TOTAL13_AD_start"] - CSF_AD["TOTAL13_AD_start"].mean())
print(CSF_AD["RID"].nunique())

1015


In [280]:
print(CSF_AD.groupby(['RID', 'VISCODE']).size().sort_values(ascending=False).head())

RID   VISCODE
3     bl         1
4399  v11        1
      v03        1
      init       1
4396  v21        1
dtype: int64


In [281]:
CSF_AD.to_csv('data/CSF_AD.csv', index=False)

In [282]:
print(sum(CSF_AD["Sleep"] == 1))

1108


In [284]:
UTIs = CSF_AD[CSF_AD["UTI"] == 1]
print(UTIs["RID"].nunique())

Sleep = CSF_AD[CSF_AD["Sleep"] == 1]
print(Sleep["RID"].nunique())
cm = CSF_AD[CSF_AD["CM"] == 1]
print(cm["RID"].nunique())
print(CSF_AD["RID"].nunique())

114
183
69
1015


In [266]:
print('--- Row / RID counts by stage ---')
print('adas_AD_dx rows:', len(adas_AD_dx), 'RID:', adas_AD_dx['RID'].nunique())
print('AD_dem rows:', len(AD_dem), 'RID:', AD_dem['RID'].nunique())
print('AD rows:', len(AD), 'RID:', AD['RID'].nunique())
print('AD_new rows:', len(AD_new), 'RID:', AD_new['RID'].nunique())
print('AD_df rows:', len(AD_df), 'RID:', AD_df['RID'].nunique())
print('AD_no_EO rows:', len(AD_no_EO), 'RID:', AD_no_EO['RID'].nunique())
print('AD_model rows:', len(AD_model), 'RID:', AD_model['RID'].nunique())
print('CSF_bl rows:', len(CSF_bl), 'RID:', CSF_bl['RID'].nunique())
print('CSF_AD rows:', len(CSF_AD), 'RID:', CSF_AD['RID'].nunique())

print('\n--- Missingness in model_vars before dropna (AD_no_EO) ---')
missing = AD_no_EO[model_vars].isna().sum().sort_values(ascending=False)
print(missing[missing > 0].head(15))

print('\n--- CSF availability among AD_model ---')
csf_rids = set(CSF_bl['RID'].dropna().unique())
ad_model_rids = set(AD_model['RID'].dropna().unique())
print('AD_model RIDs with baseline CSF:', len(ad_model_rids & csf_rids), '/', len(ad_model_rids))

print('\n--- Possible CSF duplicate baseline rows per RID ---')
dup = CSF_bl.groupby('RID').size().sort_values(ascending=False)
print('RIDs with >1 baseline row:', int((dup > 1).sum()))
print(dup.head())

--- Row / RID counts by stage ---
adas_AD_dx rows: 12955 RID: 3027
AD_dem rows: 12955 RID: 3027
AD rows: 12955 RID: 3027
AD_new rows: 12955 RID: 3027
AD_df rows: 12955 RID: 3027
AD_no_EO rows: 11152 RID: 2495
AD_model rows: 502 RID: 357
CSF_bl rows: 1621 RID: 1621
CSF_AD rows: 279 RID: 211

--- Missingness in model_vars before dropna (AD_no_EO) ---
CM                  10645
CNS                 10645
Sleep               10645
UTI                 10645
Peripheral          10645
total_conditions    10645
TOTAL13               249
CARRIER                95
HOMO                   95
days_since_entry       15
VISDATE                15
TOTAL13_AD_start        8
dtype: int64

--- CSF availability among AD_model ---
AD_model RIDs with baseline CSF: 212 / 357

--- Possible CSF duplicate baseline rows per RID ---
RIDs with >1 baseline row: 0
RID
3       1
5012    1
5027    1
5026    1
5023    1
dtype: int64
